# Final Evaluation — Social Science Concept Integration (ELSST)

**Purpose:** This notebook evaluates the calibrated pipelines on the **held-out ELSST test split**.
Each model is evaluated under **three techniques** (Clustering, Pairwise, Seeded Clustering), next to a
**character 3-gram baseline** without a language model, and subjected to the diagnostic tests:

| # | Test | What it measures |
|---|------|-----------------|
| 1 | **Embedding stability** | F1 spread, change from the noise-free run and agreement of predicted matches under embedding noise; seed variation for seeded clustering |
| 2 | **Lexical trap** | False-positive rate on lexically similar negative pairs |
| 3 | **Rare-word bias** | Recall gap between common and rare terminology, also within lexical-overlap bands |
| 4 | **Semantic decay** | Recall degradation as character overlap vanishes |
| 5 | **Structural alignment** | (A) cosine similarity against thesaurus path length among non-synonyms; (B) false-merge rate by path length |

Further outputs: F1 at a common prevalence, broader/narrower merge rates, concept-level bootstrap intervals,
top-k recall and correct matches per reviewed candidates.

> **Reproducibility:** All random operations use `SEED = 42`. Hyperparameters are fixed from the calibration on the ELSST train split (`scripts/config.py`).

## 0. Environment Setup & Imports

In [1]:
# Project root is found from the current folder, so the notebook runs both locally and on Colab.
# The device (GPU or CPU) comes from config.DEVICE and is only used to encode the terms.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value 1 overridden.*")
import sys
import os
import gc
from pathlib import Path
import importlib

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import normalize
from IPython.display import display, Markdown

# Local project modules
from scripts import model_utils_shared as shared
from scripts import model_utils_clustering as muc
from scripts import model_utils_pairwise as mup
from scripts import model_utils_seed as mus
from scripts import config

# Reload config to get latest changes
importlib.reload(config)

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {config.DEVICE}")
print(f"Project root    : {config.PROJECT_ROOT}")
print(f"Random seed     : {config.SEED}")

/home/rass/Desktop/projects/Social_Science_Construct_Harmonization/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.11.0+cpu
Device          : cpu
Project root    : /home/rass/Desktop/projects/Social_Science_Construct_Harmonization
Random seed     : 42


In [2]:
# ── Export paths ──────────────────────────────────────────────────────────────
RESULTS_DIR = str(config.RESULTS_DIRS["elsst"]) + "/"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results folder: {RESULTS_DIR}")

Results folder: /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/analysis/results/elsst/


## 1. Configuration — Best Hyperparameters (from calibration on the ELSST train split)

These parameters were selected by grid search on the ELSST train split only (no cross-validation folds).
They are **fixed** for this evaluation and must **not** be modified here.

In [3]:
# ── Models to evaluate ────────────────────────────────────────────────────────
MODELS_TO_EVAL = [
    "all-mpnet-base-v2",
    "dwulff/mpnet-personality",
    "allenai/scibert_scivocab_uncased",
    "bert-base-uncased",
]

BEST_CLUSTERING_PARAMS = config.BEST_CLUSTERING_PARAMS
BEST_PAIRWISE_THRESHOLDS = config.BEST_PAIRWISE_THRESHOLDS
BEST_SEEDED_PARAMS = config.BEST_SEEDED_PARAMS
# threshold of the character 3-gram baseline
BEST_NGRAM_THRESHOLD = config.BEST_NGRAM_THRESHOLD
BASELINE_KEY = config.BASELINE_KEY

# data set of this notebook (see config.DATASETS)
DATA_KEY = "elsst_test"

# ── Test parameters ─────────────────────────────────────────────────────────
NOISE_LEVELS = config.NOISE_LEVELS
N_RELIABILITY_RUNS = config.N_RELIABILITY_RUNS
N_SEED_DRAWS = config.N_SEED_DRAWS
N_BOOTSTRAP = config.N_BOOTSTRAP

# FAST_DEBUG = True runs the whole notebook quickly on a sample of the negative pairs and with
# fewer repetitions, to check that everything works (for example on a new Colab runtime).
# The numbers are NOT valid results. Keep it False for the real run.
FAST_DEBUG = False
if FAST_DEBUG:
    NOISE_LEVELS = [0.0, 0.1]
    N_RELIABILITY_RUNS = 2
    N_SEED_DRAWS = 2
    N_BOOTSTRAP = 50

if BEST_NGRAM_THRESHOLD is None:
    raise ValueError("Set BEST_NGRAM_THRESHOLD in scripts/config.py (printed at the end of pairwise_analysis.ipynb).")

print("✓ Configuration loaded.")

✓ Configuration loaded.


In [4]:
# Safety check: the frozen values in config.py must be the best values of the calibration
# notebooks. Prints a warning for every difference.
calib_dir = config.CALIBRATION_DIR
problems = []

pw_path = calib_dir / "pairwise_results.csv"
if pw_path.exists():
    pw = pd.read_csv(pw_path)
    for model_key in MODELS_TO_EVAL + [BASELINE_KEY]:
        sub = pw[pw["model"] == model_key]
        if sub.empty:
            problems.append(f"pairwise: no calibration rows for {model_key}")
            continue
        best_t = round(float(sub.loc[sub["f1_mean"].idxmax(), "threshold"]), 2)
        used_t = BEST_NGRAM_THRESHOLD if model_key == BASELINE_KEY else BEST_PAIRWISE_THRESHOLDS[model_key]
        if abs(best_t - used_t) > 1e-9:
            problems.append(f"pairwise {model_key}: calibration {best_t} vs config {used_t}")
else:
    problems.append("pairwise_results.csv not found")

cl_path = calib_dir / "clustering_results.csv"
if cl_path.exists():
    cl = pd.read_csv(cl_path)
    for model_key in MODELS_TO_EVAL:
        sub = cl[cl["model"] == config.MODELS[model_key]["display_name"]]
        if sub.empty:
            problems.append(f"clustering: no calibration rows for {model_key}")
            continue
        best = sub.loc[sub["f1"].idxmax()]
        used = BEST_CLUSTERING_PARAMS[model_key]
        found = {"n_components": int(best["n_components"]), "min_cluster_size": int(best["min_cluster_size"]),
                 "min_samples": int(best["min_samples"])}
        if found != used:
            problems.append(f"clustering {model_key}: calibration {found} vs config {used}")
else:
    problems.append("clustering_results.csv not found")

sd_path = calib_dir / "seed_cluster_results.csv"
if sd_path.exists():
    sd = pd.read_csv(sd_path)
    for model_key in MODELS_TO_EVAL:
        sub = sd[sd["model_name"] == model_key]
        if sub.empty:
            problems.append(f"seeded: no calibration rows for {model_key}")
            continue
        grouped = sub.groupby(["num_initial_seeds", "threshold"])["f1"].mean().reset_index()
        best = grouped.loc[grouped["f1"].idxmax()]
        found = {"n_initial_seeds": int(best["num_initial_seeds"]), "threshold": round(float(best["threshold"]), 2)}
        used = BEST_SEEDED_PARAMS[model_key]
        if found["n_initial_seeds"] != used["n_initial_seeds"] or abs(found["threshold"] - used["threshold"]) > 1e-9:
            problems.append(f"seeded {model_key}: calibration {found} vs config {used}")
else:
    problems.append("seed_cluster_results.csv not found")

if problems:
    print("config.py does not match the calibration results:")
    for p in problems:
        print("  -", p)
    if not FAST_DEBUG:
        raise ValueError("Copy the values printed by the calibration notebooks into scripts/config.py.")
else:
    print("✓ config.py matches the calibration results.")

✓ config.py matches the calibration results.


## 2. Load Data & Prepare Indices

Pairs are stored as integer term ids (`idx1`, `idx2` = row numbers in `terms`). Positive pairs come first, then negative pairs.
Negative pairs use **all labels** of both concepts (preferred labels and entry terms); pairs on the same family line
(ancestor/descendant) are excluded from the benchmark and loaded separately as broader/narrower pairs.

In [5]:
shared.setup_reproducibility(config.SEED)

data = shared.load_pair_data(**config.DATASETS[DATA_KEY])
if FAST_DEBUG:
    data = shared.subsample_negatives(data, n_negatives=200_000, seed=config.SEED)

terms = data["terms"]                 # term strings; idx1 / idx2 point into this list
unique_terms = terms
idx1 = data["idx1"]
idx2 = data["idx2"]
labels = data["label"]
shortest_path = data["shortest_path"]
pair_type = data["pair_type"]
concept1 = data["concept1"]
concept2 = data["concept2"]
pos_len = data["n_pos"]               # positives are rows [0, pos_len)
hierarchy = data["hierarchy"]
pos_rows = np.arange(len(labels)) < pos_len
neg_rows = ~pos_rows

type_counts = np.bincount(pair_type[neg_rows], minlength=3)
display(Markdown(f"""
| Split | Count |
|-------|------:|
| Positive pairs | {pos_len:,} |
| Negative pairs | {int(neg_rows.sum()):,} |
| → preferred label × preferred label | {type_counts[0]:,} |
| → preferred label × entry term | {type_counts[1]:,} |
| → entry term × entry term | {type_counts[2]:,} |
| Broader/narrower pairs (not in the benchmark) | {len(hierarchy['idx1']):,} |
"""))
print(f"Unique terms: {len(terms):,}")
print(f"Negative pairs with a thesaurus path (shortest_path ≥ 1): {int((neg_rows & (shortest_path >= 1)).sum()):,}")

# the reference prevalence for 'F1 at a common prevalence'
# is this ELSST test split; check that config.py has the right counts.
if not FAST_DEBUG:
    assert pos_len == config.REFERENCE_N_POS and int(neg_rows.sum()) == config.REFERENCE_N_NEG, \
        "Update REFERENCE_N_POS / REFERENCE_N_NEG in config.py to the ELSST test counts above."

def display_name(model_key):
    if model_key == BASELINE_KEY:
        return config.BASELINE_DISPLAY_NAME
    return config.MODELS[model_key]["display_name"]


def pair_table(rows):
    """Term strings of the given pair rows (for the example exports)."""
    return pd.DataFrame({"term1": [terms[i] for i in idx1[rows]],
                         "term2": [terms[i] for i in idx2[rows]]})

Loading pair data from /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/datasets/processed_datasets/elsst (prefix 'test_')...


  terms: 1,876 | positive pairs: 1,557 | negative pairs: 1,754,671 | broader/narrower pairs: 2,522



| Split | Count |
|-------|------:|
| Positive pairs | 1,557 |
| Negative pairs | 1,754,671 |
| → preferred label × preferred label | 530,219 |
| → preferred label × entry term | 869,119 |
| → entry term × entry term | 355,333 |
| Broader/narrower pairs (not in the benchmark) | 2,522 |


Unique terms: 1,876
Negative pairs with a thesaurus path (shortest_path ≥ 1): 819,534


## 3. Precompute Diagnostic Masks

Build boolean masks for the lexical, frequency, and difficulty subsets used by the diagnostic tests.
These are computed once and reused across all models and techniques.

In [6]:
# Character 3-gram Jaccard similarity of every term with every term, computed once with a
# sparse matrix product (no Python loop over millions of pairs).
# It defines the lexical masks and is also the score of the character 3-gram baseline.
jaccard_terms = shared.jaccard_matrix(terms, n=config.NGRAM_N)
jacc_all = shared.pair_values(jaccard_terms, idx1, idx2)

# ── Lexical trap test masks (hard & easy negatives) ──────────────────────────
# Hard negatives: lexical traps (high surface similarity, Jaccard > 0.5)
hard_neg_mask = neg_rows & (jacc_all > 0.5)
# Easy negatives: clearly distinct pairs (zero lexical overlap, Jaccard = 0.0)
easy_neg_mask = neg_rows & (jacc_all == 0.0)

# ── Semantic decay test masks (easy / hard positive) ────────────────────────
easy_pos_mask = pos_rows & (jacc_all > 0.5)
hard_pos_mask = pos_rows & (jacc_all == 0.0)
easy_mask = easy_pos_mask
hard_mask = hard_pos_mask

# ── Rare-word bias test masks (rare / common frequency) ─────────────────────
try:
    from wordfreq import zipf_frequency

    term_freq = np.zeros(len(terms))
    for i, t in enumerate(terms):
        toks = shared.simple_tokens(t)
        term_freq[i] = float(np.mean([zipf_frequency(tok, "en") for tok in toks])) if toks else 0.0

    pos_pair_freq = (term_freq[idx1[:pos_len]] + term_freq[idx2[:pos_len]]) / 2.0
    low_thr = np.quantile(pos_pair_freq, 0.10)
    high_thr = np.quantile(pos_pair_freq, 0.90)

    rare_mask = np.zeros(len(labels), dtype=bool)
    common_mask = np.zeros(len(labels), dtype=bool)
    rare_mask[:pos_len] = pos_pair_freq <= low_thr
    common_mask[:pos_len] = pos_pair_freq >= high_thr
    has_wordfreq = True
except ImportError:
    rare_mask = common_mask = np.zeros(len(labels), dtype=bool)
    has_wordfreq = False
    print("⚠ wordfreq not installed — rare-word bias test will be skipped.")

# ── Summary ──────────────────────────────────────────────────────────────────
display(Markdown(f"""
| Diagnostic Test Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | {hard_neg_mask.sum():,} |
| → Easy negatives (zero overlap, Jaccard = 0.0) | {easy_neg_mask.sum():,} |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | {easy_pos_mask.sum():,} |
| → Hard positives (Jaccard = 0.0) | {hard_pos_mask.sum():,} |
| **Frequency** | |
| → Common pairs (top 10 % freq) | {common_mask.sum():,} |
| → Rare pairs (bottom 10 % freq) | {rare_mask.sum():,} |
"""))


| Diagnostic Test Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | 137 |
| → Easy negatives (zero overlap, Jaccard = 0.0) | 1,454,151 |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | 152 |
| → Hard positives (Jaccard = 0.0) | 517 |
| **Frequency** | |
| → Common pairs (top 10 % freq) | 156 |
| → Rare pairs (bottom 10 % freq) | 156 |


## 4. Load Models & Build Embeddings

Each model is loaded once and its embeddings are cached in memory.
All subsequent technique evaluations and tests reuse these cached embeddings.

In [7]:
embedding_cache = {}

for model_name in MODELS_TO_EVAL:
    model_cfg = config.MODELS[model_name]
    model_type = model_cfg.get("type", config.TYPE_SENTENCE)

    print(f"\n{'─' * 60}")
    print(f"  {display_name(model_name)}  ({model_name})")
    print(f"{'─' * 60}")

    # model is loaded on config.DEVICE (GPU when available)
    model = shared.load_model(model_name=model_name, model_type=model_type, device=config.DEVICE)
    if model is None:
        print(f"  ✗ FAILED — skipping")
        continue

    embedding_cache[model_name] = shared.build_embeddings(model, terms, batch_size=config.BATCH_SIZE)
    print(f"  ✓ Embeddings cached — shape {embedding_cache[model_name]['norm_np'].shape}")

    # Free model from GPU memory
    del model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'═' * 60}")
print(f"  Models loaded: {len(embedding_cache)} / {len(MODELS_TO_EVAL)}")
print(f"{'═' * 60}")


────────────────────────────────────────────────────────────
  All-MPNet-Base-v2  (all-mpnet-base-v2)
────────────────────────────────────────────────────────────
Loading Model (all-mpnet-base-v2) on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12010.28it/s]


MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding 1876 unique terms...


Batches:   0%|          | 0/118 [00:00<?, ?it/s]

... (256 outputs left out)

Batches:  97%|█████████▋| 115/118 [00:08<00:00, 19.67it/s]

Batches: 100%|██████████| 118/118 [00:08<00:00, 21.55it/s]

Batches: 100%|██████████| 118/118 [00:08<00:00, 13.61it/s]

  ✓ Embeddings cached — shape (1876, 768)

════════════════════════════════════════════════════════════
  Models loaded: 4 / 4
════════════════════════════════════════════════════════════


---

## 5. Final Evaluation — Three Techniques and the Character 3-gram Baseline

We evaluate every model under three harmonisation techniques using the hyperparameters calibrated on the ELSST train split.
All predictions are stored in `preds_store` for reuse in the diagnostic sections.

In [8]:
# Storage for all predictions (used by the diagnostic tests later)
preds_store = {"clustering": {}, "pairwise": {}, "seeded": {}}


def all_pipelines():
    """(technique, model key, stored info) for every evaluated pipeline, baseline included."""
    for technique in ["clustering", "pairwise", "seeded"]:
        for model_key, info in preds_store[technique].items():
            yield technique, model_key, info

### 5.1 Clustering (UMAP + HDBSCAN)

Uses `model_utils_clustering.run_hdbscan_clustering()` and `model_utils_clustering.evaluate_clustering()` with the best parameters.

In [9]:
# ── 5.1  CLUSTERING ──────────────────────────────────────────────────────────
clustering_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    emb = embedding_cache[model_name]
    params = BEST_CLUSTERING_PARAMS[model_name]

    # Dimensionality reduction via muc.reduce_embeddings_with_umap()
    n_comp = params.get("n_components")
    if n_comp is None or n_comp >= emb["norm_np"].shape[1]:
        reduced = emb["norm_np"]
    else:
        reduced, _ = muc.reduce_embeddings_with_umap(
            emb["norm_np"], n_components=n_comp, random_state=config.SEED
        )

    # Cluster via muc.run_hdbscan_clustering()
    cluster_labels = muc.run_hdbscan_clustering(
        reduced,
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
    )

    # evaluation on the integer pair arrays
    p, r, f1, pos_acc, neg_acc, n_clusters, n_noise = muc.evaluate_clustering(
        cluster_labels, idx1, idx2, labels
    )
    preds = shared.same_cluster(cluster_labels, idx1, idx2)

    clustering_rows.append({
        "Model": display_name(model_name),
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "Clusters": n_clusters, "Noise": n_noise,
    })

    preds_store["clustering"][model_name] = {"preds": preds, "cluster_labels": cluster_labels}

display(Markdown("### Results — Clustering"))
display(pd.DataFrame(clustering_rows).set_index("Model"))

### Results — Clustering

,Precision,Recall,F1,Pos Acc,Neg Acc,Clusters,Noise
Model,,,,,,,
All-MPNet-Base-v2,0.5131,0.5029,0.5079,0.5029,0.9996,492,472
MPNet-Personality,0.4815,0.4920,0.4867,0.4920,0.9995,476,487
SciBERT (SciVocab),0.3717,0.2922,0.3272,0.2922,0.9996,421,724
BERT Base,0.2896,0.2832,0.2864,0.2832,0.9994,399,755


In [10]:
# ── Export C: cluster size distribution ───────────────────────────────────────
_sz_rows = []
for model_name in MODELS_TO_EVAL:
    info = preds_store["clustering"].get(model_name)
    if info is None:
        continue
    cl = info["cluster_labels"]
    n_noise = int((cl == -1).sum())
    n_total = int(len(cl))
    unique_cl, counts = np.unique(cl[cl != -1], return_counts=True)
    for cid, sz in zip(unique_cl, counts):
        _sz_rows.append({
            "model": display_name(model_name),
            "cluster_id": int(cid),
            "cluster_size": int(sz),
            "noise_count": n_noise,
            "total_terms": n_total,
        })
pd.DataFrame(_sz_rows).to_csv(f'{RESULTS_DIR}clustering_cluster_sizes.csv', index=False)
print(f"Saved {len(_sz_rows)} cluster-size rows")

Saved 1788 cluster-size rows


### 5.2 Pairwise (Similarity Thresholding) and Character 3-gram Baseline

Uses `model_utils_pairwise.evaluate_pairwise()` at the calibrated threshold. For the models the score is the cosine
similarity of the embeddings; for the baseline it is the character 3-gram Jaccard similarity (no language model).

In [11]:
# ── 5.2  PAIRWISE ────────────────────────────────────────────────────────────
# cosine similarity of every term pair is computed once per model as a term x term matrix
# on the CPU; the pair scores are looked up from it (fast for 60 million pairs).
pairwise_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    threshold = BEST_PAIRWISE_THRESHOLDS[model_name]
    cos_terms = shared.cosine_matrix(embedding_cache[model_name]["norm_np"])
    sims = shared.pair_values(cos_terms, idx1, idx2)
    del cos_terms

    p, r, f1, pos_acc, neg_acc, mcc = mup.evaluate_pairwise(sims, labels, threshold)
    preds = shared.above_threshold(sims, threshold).astype(np.int8)

    pairwise_rows.append({
        "Model": display_name(model_name),
        "Threshold": f"{threshold:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "MCC": f"{mcc:.4f}",
    })
    preds_store["pairwise"][model_name] = {"preds": preds, "similarities": sims, "threshold": threshold}

# character 3-gram baseline: same procedure, Jaccard instead of cosine
threshold = BEST_NGRAM_THRESHOLD
p, r, f1, pos_acc, neg_acc, mcc = mup.evaluate_pairwise(jacc_all, labels, threshold)
pairwise_rows.append({
    "Model": display_name(BASELINE_KEY),
    "Threshold": f"{threshold:.2f}",
    "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
    "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
    "MCC": f"{mcc:.4f}",
})
preds_store["pairwise"][BASELINE_KEY] = {"preds": shared.above_threshold(jacc_all, threshold).astype(np.int8),
                                         "similarities": jacc_all, "threshold": threshold}

display(Markdown("### Results — Pairwise"))
display(pd.DataFrame(pairwise_rows).set_index("Model"))

### Results — Pairwise

,Threshold,Precision,Recall,F1,Pos Acc,Neg Acc,MCC
Model,,,,,,,
All-MPNet-Base-v2,0.74,0.6948,0.3699,0.4828,0.3699,0.9999,0.5067
MPNet-Personality,0.71,0.6526,0.3475,0.4535,0.3475,0.9998,0.4759
SciBERT (SciVocab),0.90,0.3540,0.2203,0.2716,0.2203,0.9996,0.2787
BERT Base,0.91,0.2731,0.1137,0.1605,0.1137,0.9997,0.1758
Character 3-gram (baseline),0.45,0.3944,0.1368,0.2031,0.1368,0.9998,0.2319


### 5.3 Seeded Clustering

Seeds are drawn uniformly at random from all terms (no gold information). Because the result depends on the drawn seeds,
the technique is repeated with `N_SEED_DRAWS` seed draws; the table reports the mean and SD over the draws. The draw
with the median F1 is kept for the diagnostic tests.

In [12]:
# ── 5.3  SEEDED CLUSTERING ────────────────────────────────────────────────────
# seeds drawn at random from all terms (no gold information), repeated over
# N_SEED_DRAWS draws (draw d uses random seed SEED + d).
seeded_rows = []
seeded_draw_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    norm_t = embedding_cache[model_name]["norm_t"]
    params = BEST_SEEDED_PARAMS[model_name]

    draws = []
    for d in range(N_SEED_DRAWS):
        seed_idx, remaining_idx = mus.sample_initial_seeds(
            len(terms), params["n_initial_seeds"], random_state=config.SEED + d
        )
        cluster_labels = mus.seed_clustering(norm_t, seed_idx, remaining_idx, threshold=params["threshold"])
        preds = shared.same_cluster(cluster_labels, idx1, idx2)
        m = shared.binary_metrics(labels, preds)
        draws.append({"draw": d, "metrics": m, "cluster_labels": cluster_labels, "preds": preds})
        seeded_draw_rows.append({"model": display_name(model_name), "draw": d, "random_seed": config.SEED + d,
                                 "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
                                 "fpr": m["fpr"], "n_clusters": int(cluster_labels.max()) + 1})

    f1s = np.array([dr["metrics"]["f1"] for dr in draws])
    # Draw with the median F1 (lower median for an even number of draws)
    median_draw = draws[int(np.argsort(f1s)[(len(f1s) - 1) // 2])]
    mean_of = lambda key: float(np.mean([dr["metrics"][key] for dr in draws]))

    seeded_rows.append({
        "Model": display_name(model_name),
        "Seeds": params["n_initial_seeds"], "Threshold": f"{params['threshold']:.2f}",
        "Precision": f"{mean_of('precision'):.4f}", "Recall": f"{mean_of('recall'):.4f}",
        "F1": f"{mean_of('f1'):.4f}",
        "F1 SD (draws)": f"{f1s.std(ddof=1) if len(f1s) > 1 else 0.0:.4f}",
        "Pos Acc": f"{mean_of('recall'):.4f}", "Neg Acc": f"{1 - mean_of('fpr'):.4f}",
        "Median draw": median_draw["draw"],
    })

    preds_store["seeded"][model_name] = {
        "preds": median_draw["preds"], "cluster_labels": median_draw["cluster_labels"],
        "median_draw": median_draw["draw"], "draw_metrics": [dr["metrics"] for dr in draws],
    }
    del draws
    gc.collect()

pd.DataFrame(seeded_draw_rows).to_csv(f'{RESULTS_DIR}seeded_draws.csv', index=False)
display(Markdown("### Results — Seeded Clustering (mean over seed draws)"))
display(pd.DataFrame(seeded_rows).set_index("Model"))

### Results — Seeded Clustering (mean over seed draws)

,Seeds,Threshold,Precision,Recall,F1,F1 SD (draws),Pos Acc,Neg Acc,Median draw
Model,,,,,,,,,
All-MPNet-Base-v2,250,0.75,0.6852,0.3356,0.4505,0.0124,0.3356,0.9999,7
MPNet-Personality,10,0.75,0.6727,0.2748,0.3901,0.0083,0.2748,0.9999,4
SciBERT (SciVocab),500,0.90,0.2406,0.1873,0.2088,0.0195,0.1873,0.9994,3
BERT Base,500,0.90,0.1896,0.1383,0.1588,0.0147,0.1383,0.9995,2


In [13]:
# ── Export A: primary performance ─────────────────────────────────────────────
# columns besides precision, recall and F1:
#   fpr, fp_per_10k_negatives, f1_at_reference_prevalence: F1 the pipeline would
#     reach with the ELSST test prevalence ({config.REFERENCE_N_POS} positives, {config.REFERENCE_N_NEG} negatives),
#     keeping its recall and false-positive rate fixed;
#   f1_pref_pref_only: F1 when only the preferred-label negatives are counted
#     (shows how much the entry-term negatives change the numbers);
#   f1_sd_seed_draws: SD of F1 over the seed draws (seeded clustering only).
pref_pref_rows = pos_rows | (neg_rows & (pair_type == 0))
_rows = []
for technique, model_key, info in all_pipelines():
    if technique == "seeded":
        draw_metrics = info["draw_metrics"]
        m = {k: float(np.mean([dm[k] for dm in draw_metrics])) for k in ["precision", "recall", "f1", "fpr"]}
        m["fp"] = float(np.mean([dm["fp"] for dm in draw_metrics]))
        f1_sd = float(np.std([dm["f1"] for dm in draw_metrics], ddof=1)) if len(draw_metrics) > 1 else 0.0
    else:
        m = shared.binary_metrics(labels, info["preds"])
        f1_sd = np.nan
    f1_ref, prec_ref = shared.f1_at_prevalence(m["recall"], m["fpr"], config.REFERENCE_N_POS, config.REFERENCE_N_NEG)
    m_pp = shared.binary_metrics(labels[pref_pref_rows], info["preds"][pref_pref_rows])
    _rows.append({
        "technique": technique.capitalize(),
        "model": display_name(model_key),
        "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
        "f1_sd_seed_draws": f1_sd,
        "fpr": m["fpr"],
        "fp_per_10k_negatives": m["fpr"] * 10_000,
        "precision_at_reference_prevalence": prec_ref,
        "f1_at_reference_prevalence": f1_ref,
        "f1_pref_pref_only": m_pp["f1"],
    })
_perf_df = pd.DataFrame(_rows)
_perf_df.to_csv(f'{RESULTS_DIR}primary_performance.csv', index=False)
display(_perf_df.round(4))

,technique,model,precision,recall,f1,f1_sd_seed_draws,fpr,fp_per_10k_negatives,precision_at_reference_prevalence,f1_at_reference_prevalence,f1_pref_pref_only
0,Clustering,All-MPNet-Base-v2,0.5131,0.5029,0.5079,NaN,0.0004,4.2344,0.5131,0.5079,0.5859
1,Clustering,MPNet-Personality,0.4815,0.4920,0.4867,NaN,0.0005,4.7017,0.4815,0.4867,0.5725
2,Clustering,SciBERT (SciVocab),0.3717,0.2922,0.3272,NaN,0.0004,4.3826,0.3717,0.3272,0.3951
3,Clustering,BERT Base,0.2896,0.2832,0.2864,NaN,0.0006,6.1664,0.2896,0.2864,0.3568
4,Pairwise,All-MPNet-Base-v2,0.6948,0.3699,0.4828,NaN,0.0001,1.4419,0.6948,0.4828,0.5138
5,Pairwise,MPNet-Personality,0.6526,0.3475,0.4535,NaN,0.0002,1.6413,0.6526,0.4535,0.4852
6,Pairwise,SciBERT (SciVocab),0.3540,0.2203,0.2716,NaN,0.0004,3.5676,0.3540,0.2716,0.3194
7,Pairwise,BERT Base,0.2731,0.1137,0.1605,NaN,0.0003,2.6843,0.2731,0.1605,0.1814
8,Pairwise,Character 3-gram (baseline),0.3944,0.1368,0.2031,NaN,0.0002,1.8636,0.3944,0.2031,0.2215
9,Seeded,All-MPNet-Base-v2,0.6852,0.3356,0.4505,0.0124,0.0001,1.3689,0.6851,0.4506,0.4811


In [14]:
# ── Export B: confusion matrices ──────────────────────────────────────────────
_cm_rows = []
for technique, model_key, info in all_pipelines():
    tp, fp, tn, fn = shared.confusion_counts(labels, info["preds"])
    _cm_rows.append({"technique": technique.capitalize(), "model": display_name(model_key),
                     "tp": tp, "fp": fp, "fn": fn, "tn": tn})
_cm_df = pd.DataFrame(_cm_rows)
_cm_df.to_csv(f'{RESULTS_DIR}confusion_matrices.csv', index=False)
display(_cm_df)

,technique,model,tp,fp,fn,tn
0,Clustering,All-MPNet-Base-v2,783,743,774,1753928
1,Clustering,MPNet-Personality,766,825,791,1753846
2,Clustering,SciBERT (SciVocab),455,769,1102,1753902
3,Clustering,BERT Base,441,1082,1116,1753589
4,Pairwise,All-MPNet-Base-v2,576,253,981,1754418
5,Pairwise,MPNet-Personality,541,288,1016,1754383
6,Pairwise,SciBERT (SciVocab),343,626,1214,1754045
7,Pairwise,BERT Base,177,471,1380,1754200
8,Pairwise,Character 3-gram (baseline),213,327,1344,1754344
9,Seeded,All-MPNet-Base-v2,522,234,1035,1754437


In [15]:
# ── Export I: example pairs for supplementary figures ─────────────────────────
# uses the same character 3-gram masks as the tests. S16 is a histogram and S17 a threshold
# grid, because one row per pair would give files with tens of millions of rows.
_tech_names = {"clustering": "Clustering", "pairwise": "Pairwise", "seeded": "Seeded"}

def _file_name(model_key):
    return display_name(model_key).replace(" ","_").replace("(","").replace(")","").replace("/","_")

# S11: Lexical trap fooled / resisted
for technique, model_key, info in all_pipelines():
    p = info["preds"]
    for kind, rows, limit in [("fooled", np.flatnonzero(hard_neg_mask & (p == 1)), None),
                              ("resisted", np.flatnonzero(hard_neg_mask & (p == 0)), 50)]:
        rows = rows if limit is None else rows[:limit]
        _df = pair_table(rows)
        _df['jaccard'] = jacc_all[rows]
        _df['model'] = display_name(model_key)
        _df['technique'] = _tech_names[technique]
        if technique == "pairwise":
            _df['similarity'] = info["similarities"][rows]
        _df.to_csv(f'{RESULTS_DIR}s11_lexical_trap_{kind}_{_file_name(model_key)}_{_tech_names[technique]}.csv', index=False)

# S12: Semantic decay missed / caught
for technique, model_key, info in all_pipelines():
    p = info["preds"]
    for kind, rows in [("missed", np.flatnonzero(hard_pos_mask & (p == 0))),
                       ("caught", np.flatnonzero(hard_pos_mask & (p == 1)))]:
        _df = pair_table(rows)
        if technique == "pairwise":
            _df['similarity'] = info["similarities"][rows]
        _df['model'] = display_name(model_key)
        _df['technique'] = _tech_names[technique]
        _df.to_csv(f'{RESULTS_DIR}s12_semantic_decay_{kind}_{_file_name(model_key)}_{_tech_names[technique]}.csv', index=False)

# S14: Cross-model agreement (pairwise technique, language models only)
_models_pw = [m for m in MODELS_TO_EVAL if m in preds_store["pairwise"]]
_pos_preds = np.column_stack([preds_store["pairwise"][m]["preds"][:pos_len] for m in _models_pw])
_n_correct = _pos_preds.sum(axis=1)
_agreement_df = pair_table(np.arange(pos_len))
_agreement_df['n_correct'] = _n_correct
_agreement_df['category'] = np.where(_n_correct == len(_models_pw), 'unanimous_tp',
                             np.where(_n_correct == 0, 'unanimous_fn', 'disagreement'))
_agreement_df.to_csv(f'{RESULTS_DIR}s14_cross_model_agreement.csv', index=False)

# S16: Cosine distributions (All-MPNet-Base-v2, pairwise), as counts per bin of 0.01
_best_key = "all-mpnet-base-v2"
if _best_key in preds_store["pairwise"]:
    _bp = preds_store["pairwise"][_best_key]["preds"]
    _bs = preds_store["pairwise"][_best_key]["similarities"]
    _edges = np.round(np.arange(-1.0, 1.0001, 0.01), 2)
    _hist_rows = []
    for outcome, mask in [("TP", (labels == 1) & (_bp == 1)), ("FP", (labels == 0) & (_bp == 1)),
                          ("FN", (labels == 1) & (_bp == 0)), ("TN", (labels == 0) & (_bp == 0))]:
        counts, _ = np.histogram(_bs[mask], bins=_edges)
        for left, right, n in zip(_edges[:-1], _edges[1:], counts):
            _hist_rows.append({"bin_left": left, "bin_right": right, "outcome": outcome, "count": int(n)})
    pd.DataFrame(_hist_rows).to_csv(f'{RESULTS_DIR}s16_cosine_distributions.csv', index=False)

# S17: precision and recall over a threshold grid + calibrated thresholds (pairwise, all scores)
_grid = np.round(np.arange(0.0, 1.0001, 0.005), 3)
for model_key, info in preds_store["pairwise"].items():
    shared.threshold_curve(info["similarities"], labels, _grid).to_csv(
        f'{RESULTS_DIR}s17_pr_curve_{_file_name(model_key)}.csv', index=False)
    pd.DataFrame({'calibrated_threshold': [info["threshold"]]}).to_csv(
        f'{RESULTS_DIR}s17_calibrated_threshold_{_file_name(model_key)}.csv', index=False)

# S18: Error overlap (all-mpnet-base-v2)
_fn_sets = {}
for tech_key in ["clustering", "pairwise", "seeded"]:
    info = preds_store[tech_key].get(_best_key)
    if info:
        _fn_sets[tech_key] = set(np.flatnonzero(pos_rows & (info["preds"] == 0)))

if len(_fn_sets) == 3:
    cl_s, pw_s, sd_s = _fn_sets["clustering"], _fn_sets["pairwise"], _fn_sets["seeded"]
    pd.DataFrame([{
        'all_three':       len(cl_s & pw_s & sd_s),
        'clust_pair_only': len((cl_s & pw_s) - sd_s),
        'clust_seed_only': len((cl_s & sd_s) - pw_s),
        'pair_seed_only':  len((pw_s & sd_s) - cl_s),
        'clust_only':      len(cl_s - pw_s - sd_s),
        'pair_only':       len(pw_s - cl_s - sd_s),
        'seed_only':       len(sd_s - cl_s - pw_s),
    }]).to_csv(f'{RESULTS_DIR}s18_error_overlap_counts.csv', index=False)
    _all3_idx = np.array(sorted(cl_s & pw_s & sd_s), dtype=np.int64)
    pair_table(_all3_idx).sample(min(20, len(_all3_idx)), random_state=42).to_csv(
        f'{RESULTS_DIR}s18_error_overlap_irreducible.csv', index=False)

print("✓ All supplementary exports complete")

✓ All supplementary exports complete


---

## 6. Embedding stability test

**Objective:** Measure how much the output of each pipeline changes under small perturbations.

**Method:**
1. For each noise level $\sigma_\epsilon \in \{0.05, 0.1, 0.2, 0.3\}$, inject Gaussian noise: $\tilde{e} = e + \sigma_\epsilon \cdot \hat\sigma \cdot z$, where $\hat\sigma$ is the per-dimension std and $z \sim \mathcal{N}(0, 1)$, and re-run the technique `N_RELIABILITY_RUNS` times.
2. At $\sigma_\epsilon = 0$ clustering and pairwise thresholding are deterministic (one run). For seeded clustering the runs at $\sigma_\epsilon = 0$ use different **seed draws** and no noise (seed variation); the runs with noise keep the seed draw fixed (draw 0), so the two sources of variation are separated.
3. Reported separately for every level: mean F1, SD of F1 across runs, mean change of F1 from the noise-free reference run, and the **agreement of predicted matches** with the reference run (Jaccard overlap of the two sets of predicted synonym pairs: 1 = identical predictions).

The reference run is the noise-free run (for seeded clustering: draw 0 without noise).

In [16]:
# F1 variability, change from the reference run and agreement of predicted matches are
# reported separately; seed variation (seeded clustering, sigma = 0) is kept apart from
# embedding noise (sigma > 0).

def _noisy_embeddings(emb, noise_level, rng_seed):
    """L2-normalised embeddings with Gaussian noise scaled to each dimension's SD."""
    if noise_level == 0:
        return emb["norm_np"], emb["norm_t"]
    rng = np.random.default_rng(rng_seed)
    noisy = emb["raw_np"] + rng.normal(0, 1, emb["raw_np"].shape) * (noise_level * emb["sigma"])
    norm_np = np.ascontiguousarray(normalize(noisy, norm="l2", axis=1), dtype=np.float64)
    return norm_np, torch.tensor(norm_np, dtype=torch.float32)


def _predict(technique, model_name, norm_np, norm_t, seed_draw):
    """Predicted synonym pairs (boolean, one value per pair) of one run."""
    if technique == "clustering":
        params = BEST_CLUSTERING_PARAMS[model_name]
        n_comp = params.get("n_components")
        if n_comp is None or n_comp >= norm_np.shape[1]:
            reduced = norm_np
        else:
            # UMAP keeps the same random state in every run, so only the noise varies
            reduced, _ = muc.reduce_embeddings_with_umap(norm_np, n_comp, random_state=config.SEED)
        cl = muc.run_hdbscan_clustering(reduced, params["min_cluster_size"], params["min_samples"])
        return shared.same_cluster(cl, idx1, idx2).astype(bool)
    if technique == "pairwise":
        cos_terms = shared.cosine_matrix(norm_np)
        preds = shared.above_threshold(shared.pair_values(cos_terms, idx1, idx2), BEST_PAIRWISE_THRESHOLDS[model_name])
        del cos_terms
        return preds
    params = BEST_SEEDED_PARAMS[model_name]
    seed_idx, remaining_idx = mus.sample_initial_seeds(len(terms), params["n_initial_seeds"],
                                                       random_state=config.SEED + seed_draw)
    cl = mus.seed_clustering(norm_t, seed_idx, remaining_idx, threshold=params["threshold"])
    return shared.same_cluster(cl, idx1, idx2).astype(bool)


def _agreement(a, b):
    """Jaccard overlap of two sets of predicted synonym pairs (1 = identical)."""
    union = np.count_nonzero(a | b)
    return 1.0 if union == 0 else np.count_nonzero(a & b) / union


stability_records = []
for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        if model_name not in embedding_cache:
            continue
        emb = embedding_cache[model_name]
        print(f"  Stability | {technique:10s} | {model_name} ...", end="", flush=True)

        ref_preds = _predict(technique, model_name, emb["norm_np"], emb["norm_t"], seed_draw=0)
        ref_f1 = shared.binary_metrics(labels, ref_preds)["f1"]

        for level_i, nl in enumerate(NOISE_LEVELS):
            runs = []
            if nl == 0 and technique == "seeded":
                source = "seed draws"
                for d in range(N_SEED_DRAWS):
                    runs.append(ref_preds if d == 0 else
                                _predict(technique, model_name, emb["norm_np"], emb["norm_t"], seed_draw=d))
            elif nl == 0:
                source = "none (deterministic)"
                runs.append(ref_preds)
            else:
                source = "embedding noise"
                for run in range(N_RELIABILITY_RUNS):
                    norm_np_run, norm_t_run = _noisy_embeddings(emb, nl, config.SEED + 1000 * level_i + run)
                    runs.append(_predict(technique, model_name, norm_np_run, norm_t_run, seed_draw=0))

            f1_runs = np.array([shared.binary_metrics(labels, pr)["f1"] for pr in runs])
            # agreement with the reference run (the reference itself is not counted)
            agree_runs = np.array([_agreement(pr, ref_preds) for pr in runs if pr is not ref_preds] or [1.0])
            stability_records.append({
                "Technique": technique,
                "Model": display_name(model_name),
                "Noise σ": nl,
                "Variation source": source,
                "Runs": len(runs),
                "Mean F1": f1_runs.mean(),
                "SD(F1)": f1_runs.std(ddof=1) if len(runs) > 1 else 0.0,
                "Mean ΔF1 vs reference": (f1_runs - ref_f1).mean(),
                "Mean agreement": agree_runs.mean(),
                "Min agreement": agree_runs.min(),
            })
            del runs
        gc.collect()
        print(" ✓")

df_stability = pd.DataFrame(stability_records)
display(Markdown("### Embedding stability test Results"))
for tech in ["clustering", "pairwise", "seeded"]:
    display(Markdown(f"#### {tech.upper()}"))
    sub = df_stability[df_stability["Technique"] == tech].drop(columns="Technique")
    display(sub.set_index(["Model", "Noise σ"]).round(4))

  Stability | clustering | all-mpnet-base-v2 ...

 ✓
  Stability | clustering | dwulff/mpnet-personality ...

 ✓
  Stability | clustering | allenai/scibert_scivocab_uncased ...

 ✓
  Stability | clustering | bert-base-uncased ...

 ✓
  Stability | pairwise   | all-mpnet-base-v2 ...

 ✓
  Stability | pairwise   | dwulff/mpnet-personality ...

... (10 outputs left out)

#### PAIRWISE

Variation source  Runs  Mean F1  SD(F1)  \
Model              Noise σ                                                
All-MPNet-Base-v2  0.00     none (deterministic)     1   0.4828  0.0000   
                   0.05          embedding noise    10   0.4799  0.0019   
                   0.10          embedding noise    10   0.4733  0.0020   
                   0.20          embedding noise    10   0.4398  0.0020   
                   0.30          embedding noise    10   0.3824  0.0019   
MPNet-Personality  0.00     none (deterministic)     1   0.4535  0.0000   
                   0.05          embedding noise    10   0.4523  0.0014   
                   0.10          embedding noise    10   0.4453  0.0014   
                   0.20          embedding noise    10   0.4158  0.0025   
                   0.30          embedding noise    10   0.3743  0.0032   
SciBERT (SciVocab) 0.00     none (deterministic)     1   0.2716  0.0000   
                   0.05          embedding noise    10   0.2718  0.0011   
                   0.10          embedding noise    10   0.2725  0.0018   
                   0.20          embedding noise    10   0.2593  0.0035   
                   0.30          embedding noise    10   0.2128  0.0031   
BERT Base          0.00     none (deterministic)     1   0.1605  0.0000   
                   0.05          embedding noise    10   0.1577  0.0016   
                   0.10          embedding noise    10   0.1490  0.0028   
                   0.20          embedding noise    10   0.1097  0.0024   
                   0.30          embedding noise    10   0.0584  0.0031   

                            Mean ΔF1 vs reference  Mean agreement  \
Model              Noise σ                                          
All-MPNet-Base-v2  0.00                    0.0000          1.0000   
                   0.05                   -0.0029          0.9788   
                   0.10                   -0.0095          0.9436   
                   0.20                   -0.0430          0.7938   
                   0.30                   -0.1005          0.5762   
MPNet-Personality  0.00                    0.0000          1.0000   
                   0.05                   -0.0012          0.9840   
                   0.10                   -0.0082          0.9461   
                   0.20                   -0.0377          0.8055   
                   0.30                   -0.0792          0.6293   
SciBERT (SciVocab) 0.00                    0.0000          1.0000   
                   0.05                    0.0003          0.9596   
                   0.10                    0.0009          0.8779   
                   0.20                   -0.0122          0.5882   
                   0.30                   -0.0588          0.3050   
BERT Base          0.00                    0.0000          1.0000   
                   0.05                   -0.0029          0.9400   
                   0.10                   -0.0116          0.8104   
                   0.20                   -0.0508          0.3960   
                   0.30                   -0.1022          0.1085   

                            Min agreement  
Model              Noise σ                 
All-MPNet-Base-v2  0.00            1.0000  
                   0.05            0.9747  
                   0.10            0.9337  
                   0.20            0.7805  
                   0.30            0.5682  
MPNet-Personality  0.00            1.0000  
                   0.05            0.9771  
                   0.10            0.9385  
                   0.20            0.7961  
                   0.30            0.6188  
SciBERT (SciVocab) 0.00            1.0000  
                   0.05            0.9557  
                   0.10            0.8689  
                   0.20            0.5800  
                   0.30            0.2869  
BERT Base          0.00            1.0000  
                   0.05            0.9259  
                   0.10   

#### SEEDED

Variation source  Runs  Mean F1  SD(F1)  \
Model              Noise σ                                           
All-MPNet-Base-v2  0.00          seed draws    10   0.4505  0.0124   
                   0.05     embedding noise    10   0.4646  0.0025   
                   0.10     embedding noise    10   0.4560  0.0039   
                   0.20     embedding noise    10   0.4213  0.0024   
                   0.30     embedding noise    10   0.3671  0.0058   
MPNet-Personality  0.00          seed draws    10   0.3901  0.0083   
                   0.05     embedding noise    10   0.3883  0.0024   
                   0.10     embedding noise    10   0.3793  0.0024   
                   0.20     embedding noise    10   0.3510  0.0040   
                   0.30     embedding noise    10   0.2993  0.0045   
SciBERT (SciVocab) 0.00          seed draws    10   0.2088  0.0195   
                   0.05     embedding noise    10   0.2133  0.0034   
                   0.10     embedding noise    10   0.2092  0.0045   
                   0.20     embedding noise    10   0.2173  0.0051   
                   0.30     embedding noise    10   0.1985  0.0054   
BERT Base          0.00          seed draws    10   0.1588  0.0147   
                   0.05     embedding noise    10   0.1585  0.0051   
                   0.10     embedding noise    10   0.1515  0.0054   
                   0.20     embedding noise    10   0.1108  0.0049   
                   0.30     embedding noise    10   0.0701  0.0028   

                            Mean ΔF1 vs reference  Mean agreement  \
Model              Noise σ                                          
All-MPNet-Base-v2  0.00                   -0.0165          0.6771   
                   0.05                   -0.0025          0.9821   
                   0.10                   -0.0111          0.9377   
                   0.20                   -0.0457          0.7044   
                   0.30                   -0.0999          0.5288   
MPNet-Personality  0.00                   -0.0025          0.7641   
                   0.05                   -0.0043          0.9737   
                   0.10                   -0.0133          0.9296   
                   0.20                   -0.0416          0.7631   
                   0.30                   -0.0933          0.5467   
SciBERT (SciVocab) 0.00                   -0.0079          0.2892   
                   0.05                   -0.0035          0.8901   
                   0.10                   -0.0075          0.7640   
                   0.20                    0.0006          0.4569   
                   0.30                   -0.0183          0.2834   
BERT Base          0.00                   -0.0094          0.2364   
                   0.05                   -0.0097          0.7298   
                   0.10                   -0.0167          0.5741   
                   0.20                   -0.0574          0.3906   
                   0.30                   -0.0980          0.1724   

                            Min agreement  
Model              Noise σ                 
All-MPNet-Base-v2  0.00            0.6153  
                   0.05            0.9775  
                   0.10            0.9051  
                   0.20            0.6866  
                   0.30            0.5106  
MPNet-Personality  0.00            0.7341  
                   0.05            0.9562  
                   0.10            0.9096  
                   0.20            0.7511  
                   0.30            0.5234  
SciBERT (SciVocab) 0.00            0.2385  
                   0.05            0.7369  
                   0.10            0.6415  
                   0.20            0.4268  
                   0.30            0.2622  
BERT Base          0.00            0.1998  
                   0.05            0.4949  
                   0.10            0.3755  
                   0.20            0.2887  
                   0.30            0.1371

In [17]:
# ── Export D: embedding stability test ────────────────────────────────────────
_a1 = pd.DataFrame(stability_records).rename(columns={
    "Technique": "technique", "Model": "model", "Noise σ": "noise_level",
    "Variation source": "variation_source", "Runs": "n_runs",
    "Mean F1": "mean_f1", "SD(F1)": "sd_f1", "Mean ΔF1 vs reference": "mean_delta_f1",
    "Mean agreement": "mean_agreement", "Min agreement": "min_agreement",
})
_a1.to_csv(f'{RESULTS_DIR}stability.csv', index=False)
display(_a1.head(10))

,technique,model,noise_level,variation_source,n_runs,mean_f1,sd_f1,mean_delta_f1,mean_agreement,min_agreement
0,clustering,All-MPNet-Base-v2,0.00,none (deterministic),1,0.507947,0.000000,0.000000,1.000000,1.000000
1,clustering,All-MPNet-Base-v2,0.05,embedding noise,10,0.503408,0.005091,-0.004539,0.951451,0.928527
2,clustering,All-MPNet-Base-v2,0.10,embedding noise,10,0.504536,0.004785,-0.003410,0.927300,0.907077
3,clustering,All-MPNet-Base-v2,0.20,embedding noise,10,0.502162,0.007244,-0.005785,0.874926,0.862568
4,clustering,All-MPNet-Base-v2,0.30,embedding noise,10,0.507048,0.008197,-0.000899,0.818188,0.800232
5,clustering,MPNet-Personality,0.00,none (deterministic),1,0.486658,0.000000,0.000000,1.000000,1.000000
6,clustering,MPNet-Personality,0.05,embedding noise,10,0.484572,0.002929,-0.002087,0.963599,0.934258
7,clustering,MPNet-Personality,0.10,embedding noise,10,0.482586,0.004463,-0.004072,0.932702,0.891691
8,clustering,MPNet-Personality,0.20,embedding noise,10,0.481082,0.004093,-0.005576,0.878139,0.865840
9,clustering,MPNet-Personality,0.30,embedding noise,10,0.477910,0.004410,-0.008748,0.818969,0.760644


---

## 7. Lexical trap test

**Objective:** Detect whether models are fooled by surface-level orthographic similarity.

We evaluate models on two types of negative pairs (label = 0 only):

1. **Hard Negatives (Lexical Traps)**: Negative pairs whose character 3-gram Jaccard similarity exceeds 0.5 (high surface similarity but different concepts).
2. **Easy Negatives**: Negative pairs with zero lexical overlap (Jaccard = 0.0).

Because both subsets contain **only label=0 pairs**, the sole meaningful metric is:

- **FPR** (False Positive Rate): $\text{FPR} = \frac{FP}{FP + TN}$

Lower FPR on hard negatives means the model correctly rejects different concepts despite high letter overlap.
The character 3-gram baseline is included as a reference: by construction it fails on lexical traps.

In [18]:
lexical_trap_records = []

for technique, model_key, info in all_pipelines():
    preds = info["preds"]

    hard_metrics = shared.compute_subset_metrics(labels, preds, hard_neg_mask)
    easy_metrics = shared.compute_subset_metrics(labels, preds, easy_neg_mask)

    fpr_hard = hard_metrics['fpr']
    fpr_easy = easy_metrics['fpr']
    delta = fpr_hard - fpr_easy
    retention = fpr_easy / fpr_hard if fpr_hard > 0 else float("nan")

    lexical_trap_records.append({
        "Technique": technique,
        "Model": display_name(model_key),
        "FPR (Hard Neg)": f"{fpr_hard:.4f}",
        "FPR (Easy Neg)": f"{fpr_easy:.4f}",
        "ΔFPR": f"{delta:+.4f}",
        "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
        "N Hard": int(hard_metrics['n_subset']),
        "FP Hard": int(hard_metrics['fp']),
        "N Easy": int(easy_metrics['n_subset']),
        "FP Easy": int(easy_metrics['fp']),
    })

display(Markdown("### Lexical trap test Results"))
display(pd.DataFrame(lexical_trap_records).set_index(["Technique", "Model"]))

### Lexical trap test Results

FPR (Hard Neg) FPR (Easy Neg)     ΔFPR  \
Technique  Model                                                                
clustering All-MPNet-Base-v2                   0.2044         0.0002  +0.2042   
           MPNet-Personality                   0.2701         0.0001  +0.2699   
           SciBERT (SciVocab)                  0.1095         0.0002  +0.1093   
           BERT Base                           0.2482         0.0003  +0.2478   
pairwise   All-MPNet-Base-v2                   0.2482         0.0000  +0.2482   
           MPNet-Personality                   0.2263         0.0000  +0.2263   
           SciBERT (SciVocab)                  0.1241         0.0001  +0.1239   
           BERT Base                           0.0949         0.0002  +0.0946   
           Character 3-gram (baseline)         1.0000         0.0000  +1.0000   
seeded     All-MPNet-Base-v2                   0.2044         0.0000  +0.2044   
           MPNet-Personality                   0.1971         0.0000  +0.1971   
           SciBERT (SciVocab)                  0.1460         0.0003  +0.1457   
           BERT Base                           0.1095         0.0005  +0.1090   

                                       Retention Rate  N Hard  FP Hard  \
Technique  Model                                                         
clustering All-MPNet-Base-v2                   0.0008     137       28   
           MPNet-Personality                   0.0005     137       37   
           SciBERT (SciVocab)                  0.0017     137       15   
           BERT Base                           0.0014     137       34   
pairwise   All-MPNet-Base-v2                   0.0001     137       34   
           MPNet-Personality                   0.0001     137       31   
           SciBERT (SciVocab)                  0.0011     137       17   
           BERT Base                           0.0026     137       13   
           Character 3-gram (baseline)         0.0000     137      137   
seeded     All-MPNet-Base-v2                   0.0001     137       28   
           MPNet-Personality                   0.0000     137       27   
           SciBERT (SciVocab)                  0.0022     137       20   
           BERT Base                           0.0047     137       15   

                                         N Easy  FP Easy  
Technique  Model                                          
clustering All-MPNet-Base-v2            1454151      241  
           MPNet-Personality            1454151      206  
           SciBERT (SciVocab)           1454151      263  
           BERT Base                    1454151      507  
pairwise   All-MPNet-Base-v2            1454151       21  
           MPNet-Personality            1454151       18  
           SciBERT (SciVocab)           1454151      206  
           BERT Base                    1454151      363  
           Character 3-gram (baseline)  1454151        0  
seeded     All-MPNet-Base-v2            1454151       23  
           MPNet-Personality            1454151        7  
           SciBERT (SciVocab)           1454151      461  
           BERT Base                    1454151      742

In [19]:
# ── Export E: lexical trap test ───────────────────────────────────────────────
_a2 = pd.DataFrame(lexical_trap_records).rename(columns={
    "Technique": "technique", "Model": "model",
    "FPR (Hard Neg)": "hard_neg_fpr", "FPR (Easy Neg)": "easy_neg_fpr",
    "ΔFPR": "delta_fpr", "Retention Rate": "retention_rate",
    "N Hard": "n_hard", "N Easy": "n_easy",
    "FP Hard": "fp_hard", "FP Easy": "fp_easy",
})
for col in ["hard_neg_fpr", "easy_neg_fpr", "delta_fpr"]:
    _a2[col] = pd.to_numeric(_a2[col], errors='coerce')
_a2.to_csv(f'{RESULTS_DIR}lexical_trap.csv', index=False)
display(_a2)

,technique,model,hard_neg_fpr,easy_neg_fpr,delta_fpr,retention_rate,n_hard,fp_hard,n_easy,fp_easy
0,clustering,All-MPNet-Base-v2,0.2044,0.0002,0.2042,0.0008,137,28,1454151,241
1,clustering,MPNet-Personality,0.2701,0.0001,0.2699,0.0005,137,37,1454151,206
2,clustering,SciBERT (SciVocab),0.1095,0.0002,0.1093,0.0017,137,15,1454151,263
3,clustering,BERT Base,0.2482,0.0003,0.2478,0.0014,137,34,1454151,507
4,pairwise,All-MPNet-Base-v2,0.2482,0.0000,0.2482,0.0001,137,34,1454151,21
5,pairwise,MPNet-Personality,0.2263,0.0000,0.2263,0.0001,137,31,1454151,18
6,pairwise,SciBERT (SciVocab),0.1241,0.0001,0.1239,0.0011,137,17,1454151,206
7,pairwise,BERT Base,0.0949,0.0002,0.0946,0.0026,137,13,1454151,363
8,pairwise,Character 3-gram (baseline),1.0000,0.0000,1.0000,0.0000,137,137,1454151,0
9,seeded,All-MPNet-Base-v2,0.2044,0.0000,0.2044,0.0001,137,28,1454151,23


---

## 8. Rare-word bias test

**Objective:** Detect systematic bias against rare / technical terminology.

$$\Delta\text{Recall} = \text{Recall}_{\text{Common}} - \text{Recall}_{\text{Rare}}$$

- **Common pairs**: top 10 % of Zipf frequency.
- **Rare pairs**: bottom 10 % of Zipf frequency.
- A large positive $\Delta$Recall indicates the model underperforms on rare terms.

In [20]:
if not has_wordfreq:
    display(Markdown("**⚠ Skipped** — `wordfreq` library is not installed."))
else:
    rare_word_records = []
    for technique, model_key, info in all_pipelines():
        preds = info["preds"]
        rec_common = shared.recall_on_subset(labels, preds, common_mask)
        rec_rare   = shared.recall_on_subset(labels, preds, rare_mask)
        gap = rec_common - rec_rare
        retention = rec_rare / rec_common if rec_common > 0 else float("nan")
        rare_word_records.append({
            "Technique": technique,
            "Model": display_name(model_key),
            "Recall (Common)": f"{rec_common:.4f}",
            "Recall (Rare)": f"{rec_rare:.4f}",
            "ΔRecall": f"{gap:+.4f}",
            "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
            "N Common": int(common_mask.sum()),
            "N Rare": int(rare_mask.sum()),
        })

    display(Markdown("### Rare-word bias test — Results"))
    display(pd.DataFrame(rare_word_records).set_index(["Technique", "Model"]))

### Rare-word bias test — Results

Recall (Common) Recall (Rare)  ΔRecall  \
Technique  Model                                                                
clustering All-MPNet-Base-v2                    0.4615        0.4103  +0.0513   
           MPNet-Personality                    0.4744        0.3910  +0.0833   
           SciBERT (SciVocab)                   0.2628        0.2051  +0.0577   
           BERT Base                            0.3333        0.1731  +0.1603   
pairwise   All-MPNet-Base-v2                    0.3910        0.2500  +0.1410   
           MPNet-Personality                    0.3718        0.2244  +0.1474   
           SciBERT (SciVocab)                   0.1923        0.1282  +0.0641   
           BERT Base                            0.1154        0.0641  +0.0513   
           Character 3-gram (baseline)          0.2628        0.1410  +0.1218   
seeded     All-MPNet-Base-v2                    0.3526        0.2308  +0.1218   
           MPNet-Personality                    0.3269        0.1859  +0.1410   
           SciBERT (SciVocab)                   0.1603        0.0833  +0.0769   
           BERT Base                            0.1346        0.0705  +0.0641   

                                       Retention Rate  N Common  N Rare  
Technique  Model                                                         
clustering All-MPNet-Base-v2                   0.8889       156     156  
           MPNet-Personality                   0.8243       156     156  
           SciBERT (SciVocab)                  0.7805       156     156  
           BERT Base                           0.5192       156     156  
pairwise   All-MPNet-Base-v2                   0.6393       156     156  
           MPNet-Personality                   0.6034       156     156  
           SciBERT (SciVocab)                  0.6667       156     156  
           BERT Base                           0.5556       156     156  
           Character 3-gram (baseline)         0.5366       156     156  
seeded     All-MPNet-Base-v2                   0.6545       156     156  
           MPNet-Personality                   0.5686       156     156  
           SciBERT (SciVocab)                  0.5200       156     156  
           BERT Base                           0.5238       156     156

In [21]:
# ── Export F: rare-word bias test ─────────────────────────────────────────────
if 'rare_word_records' in dir() and rare_word_records:
    _a3 = pd.DataFrame(rare_word_records).rename(columns={
        "Technique": "technique", "Model": "model",
        "Recall (Common)": "recall_common", "Recall (Rare)": "recall_rare",
        "ΔRecall": "delta_recall", "Retention Rate": "retention_rate",
        "N Common": "n_common", "N Rare": "n_rare",
    })
    for col in ["recall_common", "recall_rare", "delta_recall"]:
        _a3[col] = pd.to_numeric(_a3[col], errors='coerce')
    _a3.to_csv(f'{RESULTS_DIR}rare_word.csv', index=False)
    display(_a3)
else:
    print("⚠ rare-word bias test skipped — wordfreq not installed")

,technique,model,recall_common,recall_rare,delta_recall,retention_rate,n_common,n_rare
0,clustering,All-MPNet-Base-v2,0.4615,0.4103,0.0513,0.8889,156,156
1,clustering,MPNet-Personality,0.4744,0.3910,0.0833,0.8243,156,156
2,clustering,SciBERT (SciVocab),0.2628,0.2051,0.0577,0.7805,156,156
3,clustering,BERT Base,0.3333,0.1731,0.1603,0.5192,156,156
4,pairwise,All-MPNet-Base-v2,0.3910,0.2500,0.1410,0.6393,156,156
5,pairwise,MPNet-Personality,0.3718,0.2244,0.1474,0.6034,156,156
6,pairwise,SciBERT (SciVocab),0.1923,0.1282,0.0641,0.6667,156,156
7,pairwise,BERT Base,0.1154,0.0641,0.0513,0.5556,156,156
8,pairwise,Character 3-gram (baseline),0.2628,0.1410,0.1218,0.5366,156,156
9,seeded,All-MPNet-Base-v2,0.3526,0.2308,0.1218,0.6545,156,156


### 8b. Rare-word gap within lexical-overlap bands

Rare terms may simply share fewer letters with their synonyms. To check whether the rare-word gap remains when overlap is
held constant, the positive pairs are split into three character 3-gram overlap bands (Jaccard = 0, 0 < Jaccard ≤ 0.5,
Jaccard > 0.5), and the gap Recall(Common) − Recall(Rare) is computed inside each band with the same frequency cut-offs.
The pooled gap is the average of the band gaps, weighted by $1 / (1/n_\text{rare} + 1/n_\text{common})$ of each band.

In [22]:
# rare-word gap after accounting for lexical overlap
if has_wordfreq:
    pos_jacc = jacc_all[:pos_len]
    bands = [("Jaccard = 0", pos_jacc == 0.0),
             ("0 < Jaccard ≤ 0.5", (pos_jacc > 0.0) & (pos_jacc <= 0.5)),
             ("Jaccard > 0.5", pos_jacc > 0.5)]
    rare_pos = rare_mask[:pos_len]
    common_pos = common_mask[:pos_len]

    overlap_rows = []
    for technique, model_key, info in all_pipelines():
        hits = info["preds"][:pos_len].astype(bool)
        weighted_gap, total_weight = 0.0, 0.0
        for band_name, band in bands:
            n_rare = int((band & rare_pos).sum())
            n_common = int((band & common_pos).sum())
            rec_rare = hits[band & rare_pos].mean() if n_rare > 0 else np.nan
            rec_common = hits[band & common_pos].mean() if n_common > 0 else np.nan
            gap = rec_common - rec_rare
            overlap_rows.append({"technique": technique, "model": display_name(model_key), "overlap_band": band_name,
                                 "n_rare": n_rare, "n_common": n_common, "recall_rare": rec_rare,
                                 "recall_common": rec_common, "delta_recall": gap})
            if n_rare > 0 and n_common > 0:
                weight = 1.0 / (1.0 / n_rare + 1.0 / n_common)
                weighted_gap += weight * gap
                total_weight += weight
        overlap_rows.append({"technique": technique, "model": display_name(model_key), "overlap_band": "pooled",
                             "n_rare": int(rare_pos.sum()), "n_common": int(common_pos.sum()),
                             "recall_rare": np.nan, "recall_common": np.nan,
                             "delta_recall": weighted_gap / total_weight if total_weight > 0 else np.nan})

    _a3b = pd.DataFrame(overlap_rows)
    _a3b.to_csv(f'{RESULTS_DIR}rare_word_by_overlap.csv', index=False)
    display(Markdown("### Rare-word gap within overlap bands"))
    display(_a3b.pivot_table(index=["technique", "model"], columns="overlap_band", values="delta_recall").round(4))
    display(Markdown("Pairs per band (rare / common):"))
    display(_a3b[_a3b["overlap_band"] != "pooled"].drop_duplicates("overlap_band")[["overlap_band", "n_rare", "n_common"]])

### Rare-word gap within overlap bands

overlap_band                            0 < Jaccard ≤ 0.5  Jaccard = 0  \
technique  model                                                         
clustering All-MPNet-Base-v2                      -0.1843      -0.1332   
           BERT Base                              -0.1318       0.0720   
           MPNet-Personality                      -0.2061      -0.0272   
           SciBERT (SciVocab)                     -0.2056       0.0204   
pairwise   All-MPNet-Base-v2                      -0.0815      -0.0122   
           BERT Base                              -0.0358       0.0312   
           Character 3-gram (baseline)             0.0485       0.0000   
           MPNet-Personality                      -0.0797       0.0299   
           SciBERT (SciVocab)                     -0.1150       0.0000   
seeded     All-MPNet-Base-v2                      -0.1119       0.0095   
           BERT Base                               0.0059       0.0312   
           MPNet-Personality                      -0.0883       0.0312   
           SciBERT (SciVocab)                     -0.0489       0.0000   

overlap_band                            Jaccard > 0.5  pooled  
technique  model                                               
clustering All-MPNet-Base-v2                  -0.0119 -0.1372  
           BERT Base                           0.4524  0.0390  
           MPNet-Personality                  -0.0119 -0.1092  
           SciBERT (SciVocab)                  0.0317 -0.0845  
pairwise   All-MPNet-Base-v2                   0.0794 -0.0297  
           BERT Base                           0.0794  0.0076  
           Character 3-gram (baseline)         0.0000  0.0229  
           MPNet-Personality                   0.1349 -0.0044  
           SciBERT (SciVocab)                  0.1270 -0.0332  
seeded     All-MPNet-Base-v2                   0.0437 -0.0422  
           BERT Base                          -0.0119  0.0121  
           MPNet-Personality                   0.1548 -0.0047  
           SciBERT (SciVocab)                  0.1865  0.0079

Pairs per band (rare / common):

,overlap_band,n_rare,n_common
0,Jaccard = 0,92,32
1,0 < Jaccard ≤ 0.5,46,96
2,Jaccard > 0.5,18,28


---

## 9. Semantic decay test

**Objective:** Test robustness as keyword overlap vanishes.

$$\Delta\text{Recall} = \text{Recall}_{\text{Hard}} - \text{Recall}_{\text{Easy}}$$

- **Easy positives**: Jaccard > 0.5 (lexical anchors).
- **Hard positives**: Jaccard = 0.0 (no character overlap; only meaning can connect the terms).
- A negative ΔRecall indicates performance degrades on purely-semantic pairs.

**Retention Rate** = Recall(Hard) / Recall(Easy). Closer to 1.0 = less semantic decay. The character 3-gram baseline is
included as a reference: by construction it cannot recover hard positives.

In [23]:
semantic_decay_records = []

for technique, model_key, info in all_pipelines():
    preds = info["preds"]
    rec_easy = shared.recall_on_subset(labels, preds, easy_mask)
    rec_hard = shared.recall_on_subset(labels, preds, hard_mask)
    delta = rec_hard - rec_easy
    retention = rec_hard / rec_easy if rec_easy > 0 else float("nan")
    semantic_decay_records.append({
        "Technique": technique,
        "Model": display_name(model_key),
        "Recall (Easy)": f"{rec_easy:.4f}",
        "Recall (Hard)": f"{rec_hard:.4f}",
        "ΔRecall": f"{delta:+.4f}",
        "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
        "N Easy": int(easy_mask.sum()),
        "N Hard": int(hard_mask.sum()),
    })

display(Markdown("### Semantic decay test Results"))
display(pd.DataFrame(semantic_decay_records).set_index(["Technique", "Model"]))

### Semantic decay test Results

Recall (Easy) Recall (Hard)  ΔRecall  \
Technique  Model                                                              
clustering All-MPNet-Base-v2                  0.8684        0.2708  -0.5976   
           MPNet-Personality                  0.8816        0.2495  -0.6321   
           SciBERT (SciVocab)                 0.6447        0.0600  -0.5848   
           BERT Base                          0.6053        0.0812  -0.5240   
pairwise   All-MPNet-Base-v2                  0.8618        0.0909  -0.7709   
           MPNet-Personality                  0.8553        0.0716  -0.7837   
           SciBERT (SciVocab)                 0.5329        0.0251  -0.5077   
           BERT Base                          0.3355        0.0368  -0.2988   
           Character 3-gram (baseline)        1.0000        0.0000  -1.0000   
seeded     All-MPNet-Base-v2                  0.7895        0.0890  -0.7005   
           MPNet-Personality                  0.8158        0.0464  -0.7694   
           SciBERT (SciVocab)                 0.4474        0.0251  -0.4222   
           BERT Base                          0.3553        0.0368  -0.3185   

                                       Retention Rate  N Easy  N Hard  
Technique  Model                                                       
clustering All-MPNet-Base-v2                   0.3118     152     517  
           MPNet-Personality                   0.2830     152     517  
           SciBERT (SciVocab)                  0.0930     152     517  
           BERT Base                           0.1342     152     517  
pairwise   All-MPNet-Base-v2                   0.1055     152     517  
           MPNet-Personality                   0.0837     152     517  
           SciBERT (SciVocab)                  0.0472     152     517  
           BERT Base                           0.1095     152     517  
           Character 3-gram (baseline)         0.0000     152     517  
seeded     All-MPNet-Base-v2                   0.1127     152     517  
           MPNet-Personality                   0.0569     152     517  
           SciBERT (SciVocab)                  0.0562     152     517  
           BERT Base                           0.1034     152     517

In [24]:
# ── Export G: semantic decay test ─────────────────────────────────────────────
_a4 = pd.DataFrame(semantic_decay_records).rename(columns={
    "Technique": "technique", "Model": "model",
    "Recall (Easy)": "recall_easy", "Recall (Hard)": "recall_hard",
    "ΔRecall": "delta_recall", "Retention Rate": "retention_rate",
    "N Easy": "n_easy", "N Hard": "n_hard",
})
for col in ["recall_easy", "recall_hard", "delta_recall"]:
    _a4[col] = pd.to_numeric(_a4[col], errors='coerce')
_a4.to_csv(f'{RESULTS_DIR}semantic_decay.csv', index=False)
display(_a4)

,technique,model,recall_easy,recall_hard,delta_recall,retention_rate,n_easy,n_hard
0,clustering,All-MPNet-Base-v2,0.8684,0.2708,-0.5976,0.3118,152,517
1,clustering,MPNet-Personality,0.8816,0.2495,-0.6321,0.2830,152,517
2,clustering,SciBERT (SciVocab),0.6447,0.0600,-0.5848,0.0930,152,517
3,clustering,BERT Base,0.6053,0.0812,-0.5240,0.1342,152,517
4,pairwise,All-MPNet-Base-v2,0.8618,0.0909,-0.7709,0.1055,152,517
5,pairwise,MPNet-Personality,0.8553,0.0716,-0.7837,0.0837,152,517
6,pairwise,SciBERT (SciVocab),0.5329,0.0251,-0.5077,0.0472,152,517
7,pairwise,BERT Base,0.3355,0.0368,-0.2988,0.1095,152,517
8,pairwise,Character 3-gram (baseline),1.0000,0.0000,-1.0000,0.0000,152,517
9,seeded,All-MPNet-Base-v2,0.7895,0.0890,-0.7005,0.1127,152,517


---

## 10. Structural alignment test

**Objective:** A measurement instrument satisfies the structural aspect of construct validity when the internal organisation
it implies matches the internal organisation of the construct (Messick, 1995). Two parts:

**(A) Model level — similarity against thesaurus distance.** Among non-synonym pairs that are connected in the thesaurus
graph (path length ≥ 2; direct parent–child pairs are not negatives), we correlate the similarity score with the
shortest-path length (Spearman ρ). A negative ρ means similarity falls as terms get further apart, i.e. the embedding space
follows the taxonomy. Synonym decisions play no role, so false merges cannot raise this score. One value per model
(and the character 3-gram baseline as a reference).

**(B) Pipeline level — false merges by thesaurus distance.** For every pipeline, the false-positive rate among negative
pairs at each path length. Lower is better at every distance. `near_miss_share` is the share of all false merges that
join concepts at path length 2–4 (close but different concepts).

In [25]:
# A correlation between the binary synonym decision and graph proximity could rise when a
# pipeline wrongly merges nearby concepts. We therefore use part A (continuous similarity vs path
# length among non-synonyms) and part B (false-merge rate by path length per pipeline).

# ── Part A: similarity vs path length among connected non-synonym pairs ──────
connected_neg = neg_rows & (shortest_path >= 2)
path_values = shortest_path[connected_neg].astype(float)
print(f"Connected non-synonym pairs: {int(connected_neg.sum()):,}")

# On a large thesaurus the rank correlation is computed on a fixed random sample of pairs
# (5 million pairs give a standard error of about 0.0005).
SPEARMAN_MAX_PAIRS = 5_000_000
DIST_BANDS = [("2", 2, 2), ("3", 3, 3), ("4", 4, 4), ("5-6", 5, 6), ("7+", 7, 10_000)]

structural_records = []
similarity_by_distance = []
for model_key, info in preds_store["pairwise"].items():
    sims_conn = info["similarities"][connected_neg]
    rho = shared.spearman_corr(sims_conn, path_values, max_n=SPEARMAN_MAX_PAIRS, seed=config.SEED)
    structural_records.append({"model": display_name(model_key), "spearman_rho": rho,
                               "n_pairs": int(connected_neg.sum())})
    for band_name, lo, hi in DIST_BANDS:
        in_band = (path_values >= lo) & (path_values <= hi)
        similarity_by_distance.append({"model": display_name(model_key), "path_length": band_name,
                                       "n_pairs": int(in_band.sum()),
                                       "mean_similarity": float(sims_conn[in_band].mean()) if in_band.any() else np.nan})
    del sims_conn

display(Markdown("### (A) Similarity vs thesaurus path length (Spearman ρ; negative = follows the taxonomy)"))
display(pd.DataFrame(structural_records).set_index("model").round(4))
display(pd.DataFrame(similarity_by_distance).pivot(index="model", columns="path_length", values="mean_similarity").round(4))

# ── Part B: false-merge rate by path length for every pipeline ───────────────
false_merge_records = []
for technique, model_key, info in all_pipelines():
    fp_neg = neg_rows & (info["preds"] == 1)
    row = {"technique": technique, "model": display_name(model_key)}
    for band_name, lo, hi in DIST_BANDS:
        band = neg_rows & (shortest_path >= lo) & (shortest_path <= hi)
        row[f"fpr_path_{band_name}"] = (fp_neg & band).sum() / band.sum() if band.any() else np.nan
    unconnected = neg_rows & (shortest_path < 0)
    row["fpr_not_connected"] = (fp_neg & unconnected).sum() / unconnected.sum() if unconnected.any() else np.nan
    n_fp = int(fp_neg.sum())
    near = fp_neg & (shortest_path >= 2) & (shortest_path <= 4)
    row["n_false_merges"] = n_fp
    row["near_miss_share"] = near.sum() / n_fp if n_fp > 0 else np.nan
    false_merge_records.append(row)

display(Markdown("### (B) False-merge rate by thesaurus path length (lower is better)"))
display(pd.DataFrame(false_merge_records).set_index(["technique", "model"]))

Connected non-synonym pairs: 819,534


### (A) Similarity vs thesaurus path length (Spearman ρ; negative = follows the taxonomy)

,spearman_rho,n_pairs
model,,
All-MPNet-Base-v2,-0.2214,819534
MPNet-Personality,-0.1344,819534
SciBERT (SciVocab),-0.1499,819534
BERT Base,-0.1795,819534
Character 3-gram (baseline),-0.0293,819534


path_length,2,3,4,5-6,7+
model,,,,,
All-MPNet-Base-v2,0.4218,0.3263,0.2721,0.2228,0.1520
BERT Base,0.7029,0.6652,0.6494,0.6403,0.6059
Character 3-gram (baseline),0.0897,0.0363,0.0269,0.0196,0.0123
MPNet-Personality,0.3419,0.2452,0.2009,0.1642,0.1178
SciBERT (SciVocab),0.7606,0.7324,0.7213,0.7046,0.6821


### (B) False-merge rate by thesaurus path length (lower is better)

fpr_path_2  fpr_path_3  fpr_path_4  \
technique  model                                                             
clustering All-MPNet-Base-v2              0.055272    0.009708    0.003056   
           MPNet-Personality              0.043651    0.007438    0.003205   
           SciBERT (SciVocab)             0.043651    0.006682    0.003354   
           BERT Base                      0.049603    0.010590    0.003578   
pairwise   All-MPNet-Base-v2              0.019274    0.004413    0.000820   
           MPNet-Personality              0.019558    0.004034    0.001193   
           SciBERT (SciVocab)             0.031746    0.005925    0.002236   
           BERT Base                      0.011054    0.001387    0.000671   
           Character 3-gram (baseline)    0.015590    0.002900    0.001565   
seeded     All-MPNet-Base-v2              0.016156    0.004665    0.000447   
           MPNet-Personality              0.012472    0.003404    0.000820   
           SciBERT (SciVocab)             0.032313    0.005169    0.003802   
           BERT Base                      0.016440    0.003404    0.001044   

                                        fpr_path_5-6  fpr_path_7+  \
technique  model                                                    
clustering All-MPNet-Base-v2                0.001185     0.000197   
           MPNet-Personality                0.001849     0.000314   
           SciBERT (SciVocab)               0.001683     0.000242   
           BERT Base                        0.001422     0.000379   
pairwise   All-MPNet-Base-v2                0.000521     0.000073   
           MPNet-Personality                0.000616     0.000086   
           SciBERT (SciVocab)               0.001114     0.000183   
           BERT Base                        0.000545     0.000197   
           Character 3-gram (baseline)      0.000569     0.000133   
seeded     All-MPNet-Base-v2                0.000427     0.000076   
           MPNet-Personality                0.000403     0.000073   
           SciBERT (SciVocab)               0.001778     0.000308   
           BERT Base                        0.000948     0.000440   

                                        fpr_not_connected  n_false_merges  \
technique  model                                                            
clustering All-MPNet-Base-v2                     0.000248             743   
           MPNet-Personality                     0.000273             825   
           SciBERT (SciVocab)                    0.000282             769   
           BERT Base                             0.000460            1082   
pairwise   All-MPNet-Base-v2                     0.000066             253   
           MPNet-Personality                     0.000086             288   
           SciBERT (SciVocab)                    0.000269             626   
           BERT Base                             0.000258             471   
           Character 3-gram (baseline)           0.000111             327   
seeded     All-MPNet-Base-v2                     0.000063             234   
           MPNet-Personality                     0.000066             216   
           SciBERT (SciVocab)                    0.000418             904   
           BERT Base                             0.000509             946   

                                        near_miss_share  
technique  model                                         
clustering All-MPNet-Base-v2                   0.421265  
           MPNet-Personality                   0.310303  
           SciBERT (SciVocab)                  0.327698  
           BERT Base                           0.283734  
pairwise   All-MPNet-Base-v2                   0.450593  
           MPNet-Personality                   0.406250  
           SciBERT (SciVocab)                  0.301917  
           BERT Base                           0.125265  
           Character 3-gram (baseline)         0.302752  
seeded     All-MPNet-Base-v2           

In [26]:
# ── Export H: structural alignment test ───────────────────────────────────────
pd.DataFrame(structural_records).to_csv(f'{RESULTS_DIR}structural_alignment.csv', index=False)
pd.DataFrame(similarity_by_distance).to_csv(f'{RESULTS_DIR}structural_similarity_by_distance.csv', index=False)
pd.DataFrame(false_merge_records).to_csv(f'{RESULTS_DIR}structural_false_merge_by_distance.csv', index=False)
print("✓ Structural alignment exports saved")

✓ Structural alignment exports saved


In [27]:
# ── Export J: structural alignment examples ────────────────────────────────────
_rng = np.random.default_rng(42)
_struct_rows = []
for dist in [2, 3, 4, 5]:
    rows = np.flatnonzero(neg_rows & (shortest_path == dist))
    rows = np.sort(_rng.choice(rows, size=min(5, len(rows)), replace=False)) if len(rows) else rows
    _sub = pair_table(rows)
    _sub['shortest_path'] = dist
    if _best_key in preds_store['pairwise']:
        _sub['cosine_sim'] = preds_store['pairwise'][_best_key]['similarities'][rows]
    _struct_rows.append(_sub)
pd.concat(_struct_rows).to_csv(f'{RESULTS_DIR}s15_structural_examples.csv', index=False)
print("✓ Structural examples saved")

✓ Structural examples saved


---

## 11. Broader/narrower pairs (supplementary)

Ancestor–descendant pairs are excluded from the benchmark negatives, because a broader and a narrower term share part of
their meaning. Here we report how often each pipeline merges a concept with one of its ancestors, by depth
(1 = direct parent, 2 = grandparent, 3+ = more distant ancestor). These pairs never enter Table 1.

In [28]:
# merge rate of broader/narrower pairs, reported separately
h_idx1, h_idx2, h_depth = hierarchy["idx1"], hierarchy["idx2"], hierarchy["depth"]
depth_bands = [("1", h_depth == 1), ("2", h_depth == 2), ("3+", h_depth >= 3), ("all", np.ones(len(h_depth), dtype=bool))]

hierarchy_records = []
for technique, model_key, info in all_pipelines():
    if technique in ("clustering", "seeded"):
        h_pred = shared.same_cluster(info["cluster_labels"], h_idx1, h_idx2).astype(bool)
    elif model_key == BASELINE_KEY:
        h_pred = shared.above_threshold(shared.pair_values(jaccard_terms, h_idx1, h_idx2), info["threshold"])
    else:
        norm = embedding_cache[model_key]["norm_np"]
        h_cos = np.einsum("ij,ij->i", norm[h_idx1], norm[h_idx2]).astype(np.float32)
        h_pred = shared.above_threshold(h_cos, info["threshold"])
    row = {"technique": technique, "model": display_name(model_key)}
    for band_name, band in depth_bands:
        row[f"merge_rate_depth_{band_name}"] = h_pred[band].mean() if band.any() else np.nan
        row[f"n_depth_{band_name}"] = int(band.sum())
    row["fpr_benchmark_negatives"] = shared.binary_metrics(labels, info["preds"])["fpr"]
    hierarchy_records.append(row)

_hier = pd.DataFrame(hierarchy_records)
_hier.to_csv(f'{RESULTS_DIR}hierarchy_merge_rate.csv', index=False)
display(_hier.set_index(["technique", "model"]).round(4))

merge_rate_depth_1  n_depth_1  \
technique  model                                                        
clustering All-MPNet-Base-v2                        0.1572        986   
           MPNet-Personality                        0.1329        986   
           SciBERT (SciVocab)                       0.0700        986   
           BERT Base                                0.0822        986   
pairwise   All-MPNet-Base-v2                        0.0984        986   
           MPNet-Personality                        0.0842        986   
           SciBERT (SciVocab)                       0.0517        986   
           BERT Base                                0.0274        986   
           Character 3-gram (baseline)              0.0588        986   
seeded     All-MPNet-Base-v2                        0.0923        986   
           MPNet-Personality                        0.0568        986   
           SciBERT (SciVocab)                       0.0477        986   
           BERT Base                                0.0456        986   

                                        merge_rate_depth_2  n_depth_2  \
technique  model                                                        
clustering All-MPNet-Base-v2                        0.0175        684   
           MPNet-Personality                        0.0278        684   
           SciBERT (SciVocab)                       0.0073        684   
           BERT Base                                0.0219        684   
pairwise   All-MPNet-Base-v2                        0.0058        684   
           MPNet-Personality                        0.0015        684   
           SciBERT (SciVocab)                       0.0044        684   
           BERT Base                                0.0058        684   
           Character 3-gram (baseline)              0.0219        684   
seeded     All-MPNet-Base-v2                        0.0102        684   
           MPNet-Personality                        0.0044        684   
           SciBERT (SciVocab)                       0.0088        684   
           BERT Base                                0.0117        684   

                                        merge_rate_depth_3+  n_depth_3+  \
technique  model                                                          
clustering All-MPNet-Base-v2                         0.0035         852   
           MPNet-Personality                         0.0059         852   
           SciBERT (SciVocab)                        0.0000         852   
           BERT Base                                 0.0023         852   
pairwise   All-MPNet-Base-v2                         0.0012         852   
           MPNet-Personality                         0.0012         852   
           SciBERT (SciVocab)                        0.0012         852   
           BERT Base                                 0.0000         852   
           Character 3-gram (baseline)               0.0153         852   
seeded     All-MPNet-Base-v2                         0.0012         852   
           MPNet-Personality                         0.0000         852   
           SciBERT (SciVocab)                        0.0047         852   
           BERT Base                                 0.0012         852   

                                        merge_rate_depth_all  n_depth_all  \
technique  model                                                            
clustering All-MPNet-Base-v2                          0.0674         2522   
           MPNet-Personality                          0.0615         2522   
           SciBERT (SciVocab)                         0.0293         2522   
           BERT Base                                  0.0389         2522   
pairwise   All-MPNet-Base-v2                          0.0404         2522   
           MPNet-Personality                          0.0337         2522   
           SciBERT (SciVocab)                         0.0218         2522   
           BERT Base         

---

## 12. Uncertainty — concept-level bootstrap

Pairs are not independent: one concept appears in many pairs. We therefore resample **concepts** with replacement
(`N_BOOTSTRAP` samples). A concept drawn k times counts k times; a pair of two different concepts gets weight
k₁·k₂, a pair inside one concept weight k. Metrics are recomputed in every sample and the 2.5th and 97.5th percentiles give
95 % intervals. The same bootstrap samples are used for every pipeline, so the F1 differences between pipelines also get
intervals (paired comparison). The pipelines are kept fixed (for seeded clustering: the median draw); seed and noise
variation are reported in Sections 5.3 and 6.

In [29]:
# 95 % intervals for every pipeline
# and for the F1 differences between pipelines, from one set of concept-level bootstrap samples.
concept_ids, boot_w = shared.bootstrap_weights(concept1, concept2, n_boot=N_BOOTSTRAP, seed=config.SEED)
col1 = np.searchsorted(concept_ids, concept1).astype(np.int32)
col2 = np.searchsorted(concept_ids, concept2).astype(np.int32)
print(f"Concepts resampled: {len(concept_ids):,} | bootstrap samples: {N_BOOTSTRAP}")

subset_definitions = [
    ("fpr_hard_negatives", hard_neg_mask),
    ("recall_easy_positives", easy_pos_mask),
    ("recall_hard_positives", hard_pos_mask),
    ("recall_common", common_mask),
    ("recall_rare", rare_mask),
]

interval_rows = []
boot_f1 = {}
for technique, model_key, info in all_pipelines():
    preds = info["preds"]
    name = f"{technique} | {display_name(model_key)}"
    point = shared.binary_metrics(labels, preds)
    run = "median draw" if technique == "seeded" else "single run"
    samples = shared.bootstrap_prf(boot_w, col1, col2, labels, preds)
    boot_f1[name] = (point["f1"], samples["f1"])
    for metric in ["precision", "recall", "f1"]:
        low, high = shared.percentile_interval(samples[metric])
        interval_rows.append({"technique": technique, "model": display_name(model_key), "run": run,
                              "metric": metric, "estimate": point[metric], "ci_low": low, "ci_high": high})
    for metric, mask in subset_definitions:
        if not mask.any():
            continue
        rate_samples = shared.bootstrap_rate(boot_w, col1, col2, mask, preds)
        low, high = shared.percentile_interval(rate_samples)
        interval_rows.append({"technique": technique, "model": display_name(model_key), "run": run,
                              "metric": metric, "estimate": float(preds[mask].mean()), "ci_low": low, "ci_high": high,
                              "n_pairs": int(mask.sum())})

_intervals = pd.DataFrame(interval_rows)
_intervals.to_csv(f'{RESULTS_DIR}bootstrap_intervals.csv', index=False)
display(Markdown("### F1 with 95 % concept-level bootstrap intervals"))
display(_intervals[_intervals["metric"] == "f1"].set_index(["technique", "model"]).round(4))

# Paired F1 differences between all pipelines
diff_rows = []
names = list(boot_f1)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        diff = boot_f1[a][1] - boot_f1[b][1]
        low, high = shared.percentile_interval(diff)
        diff_rows.append({"pipeline_a": a, "pipeline_b": b,
                          "f1_difference": boot_f1[a][0] - boot_f1[b][0],
                          "ci_low": low, "ci_high": high,
                          "share_samples_a_better": float(np.mean(diff > 0))})
_diffs = pd.DataFrame(diff_rows)
_diffs.to_csv(f'{RESULTS_DIR}bootstrap_f1_differences.csv', index=False)
print(f"✓ {len(_diffs)} pairwise F1 differences saved (interval excluding 0 = difference beyond sampling noise)")

Concepts resampled: 1,031 | bootstrap samples: 1000


### F1 with 95 % concept-level bootstrap intervals

run metric  estimate  ci_low  \
technique  model                                                               
clustering All-MPNet-Base-v2             single run     f1    0.5079  0.4686   
           MPNet-Personality             single run     f1    0.4867  0.4410   
           SciBERT (SciVocab)            single run     f1    0.3272  0.2878   
           BERT Base                     single run     f1    0.2864  0.2482   
pairwise   All-MPNet-Base-v2             single run     f1    0.4828  0.4381   
           MPNet-Personality             single run     f1    0.4535  0.4116   
           SciBERT (SciVocab)            single run     f1    0.2716  0.2370   
           BERT Base                     single run     f1    0.1605  0.1326   
           Character 3-gram (baseline)   single run     f1    0.2031  0.1740   
seeded     All-MPNet-Base-v2            median draw     f1    0.4514  0.4002   
           MPNet-Personality            median draw     f1    0.3918  0.3491   
           SciBERT (SciVocab)           median draw     f1    0.2115  0.1778   
           BERT Base                    median draw     f1    0.1562  0.1263   

                                        ci_high  n_pairs  
technique  model                                          
clustering All-MPNet-Base-v2             0.5492      NaN  
           MPNet-Personality             0.5267      NaN  
           SciBERT (SciVocab)            0.3700      NaN  
           BERT Base                     0.3256      NaN  
pairwise   All-MPNet-Base-v2             0.5253      NaN  
           MPNet-Personality             0.4942      NaN  
           SciBERT (SciVocab)            0.3061      NaN  
           BERT Base                     0.1906      NaN  
           Character 3-gram (baseline)   0.2389      NaN  
seeded     All-MPNet-Base-v2             0.5063      NaN  
           MPNet-Personality             0.4359      NaN  
           SciBERT (SciVocab)            0.2491      NaN  
           BERT Base                     0.1909      NaN

✓ 78 pairwise F1 differences saved (interval excluding 0 = difference beyond sampling noise)


---

## 13. Top-k recall and correct matches per reviewed candidates

**Top-k recall:** for every synonym pair (in both directions), is the synonym among the k nearest neighbours of the other
term among all terms? Ties are broken at random in expectation (a synonym tied with others counts as the
share of tie orders that put it inside the top k). This separates synonyms that are ranked poorly from synonyms that are ranked
well but rejected by the threshold.

**Correct matches per reviewed candidates:** all pairs are ranked by similarity; we count the synonym pairs among the first
100, 500 and 1,000 pairs, as a curator reviewing the top of the list would see them (ties broken at random, fixed seed).
Only benchmark pairs are ranked; broader/narrower pairs are not in this list.

In [30]:
# top-k recall and correct matches per reviewed candidates
pos_q = np.concatenate([idx1[:pos_len], idx2[:pos_len]])
pos_t = np.concatenate([idx2[:pos_len], idx1[:pos_len]])

ranking_rows = []
for model_key, info in preds_store["pairwise"].items():
    if model_key == BASELINE_KEY:
        score_terms = jaccard_terms
    else:
        score_terms = shared.cosine_matrix(embedding_cache[model_key]["norm_np"])
    topk = shared.topk_recall(score_terms, pos_q, pos_t, config.TOPK_LIST)
    correct = shared.correct_in_top(info["similarities"], labels, config.REVIEW_BUDGETS, seed=config.SEED)
    row = {"model": display_name(model_key)}
    row.update({f"recall_at_{k}": v for k, v in topk.items()})
    row.update({f"correct_in_top_{n}": v for n, v in correct.items()})
    ranking_rows.append(row)
    if model_key != BASELINE_KEY:
        del score_terms
    gc.collect()

_ranking = pd.DataFrame(ranking_rows)
_ranking.to_csv(f'{RESULTS_DIR}topk_and_review.csv', index=False)
display(_ranking.set_index("model").round(4))

,recall_at_1,recall_at_5,recall_at_10,correct_in_top_100,correct_in_top_500,correct_in_top_1000
model,,,,,,
All-MPNet-Base-v2,0.3179,0.7065,0.8128,99,405,638
MPNet-Personality,0.3131,0.6952,0.7964,99,379,611
SciBERT (SciVocab),0.2091,0.4416,0.5186,85,256,349
BERT Base,0.2155,0.4473,0.5222,55,146,232
Character 3-gram (baseline),0.1608,0.3750,0.4734,72,205,293


---

## 14. Consolidated Summary

A single overview table combining the core F1 metric across all models and techniques, plus a per-technique best-model highlight.

In [31]:
# ── Build consolidated F1 summary ─────────────────────────────────────────────
summary_rows = []

for tech_name, rows_list in [("Clustering", clustering_rows),
                              ("Pairwise", pairwise_rows),
                              ("Seeded", seeded_rows)]:
    for row in rows_list:
        summary_rows.append({
            "Technique": tech_name,
            "Model": row["Model"],
            "Precision": row["Precision"],
            "Recall": row["Recall"],
            "F1": row["F1"],
        })

df_summary = pd.DataFrame(summary_rows)

display(Markdown("### F1 Summary — All Models × All Techniques"))
pivot_f1 = df_summary.pivot(index="Model", columns="Technique", values="F1")
display(pivot_f1)

# ── Best model per technique ─────────────────────────────────────────────────
display(Markdown("### Best Model per Technique"))
for tech in ["Clustering", "Pairwise", "Seeded"]:
    sub = df_summary[df_summary["Technique"] == tech]
    best = sub.loc[sub["F1"].astype(float).idxmax()]
    print(f"  {tech:12s} → {best['Model']}  (F1 = {best['F1']})")

# ── Save summary to CSV ─────────────────────────────────────────────────────
summary_path = f"{RESULTS_DIR}final_test_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to {summary_path}")

### F1 Summary — All Models × All Techniques

Technique,Clustering,Pairwise,Seeded
Model,,,
All-MPNet-Base-v2,0.5079,0.4828,0.4505
BERT Base,0.2864,0.1605,0.1588
Character 3-gram (baseline),NaN,0.2031,NaN
MPNet-Personality,0.4867,0.4535,0.3901
SciBERT (SciVocab),0.3272,0.2716,0.2088


### Best Model per Technique

  Clustering   → All-MPNet-Base-v2  (F1 = 0.5079)
  Pairwise     → All-MPNet-Base-v2  (F1 = 0.4828)
  Seeded       → All-MPNet-Base-v2  (F1 = 0.4505)

✓ Summary saved to /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/analysis/results/elsst/final_test_summary.csv


---

## Reproducibility Notes

| Item | Value |
|------|-------|
| Random seed | `SEED = 42` across all operations (seed draw d uses `SEED + d`) |
| Data | `datasets/processed_datasets/elsst/test_*.parquet`, built by `scripts/process_dataset.py` |
| Device | `config.DEVICE` (GPU only for encoding; similarities, clustering and metrics on CPU) |
| Hyperparameters | Fixed from the calibration on the ELSST train split (`scripts/config.py`) |
| Uncertainty | Concept-level bootstrap, `N_BOOTSTRAP` samples |

**To reproduce:** Run all cells top-to-bottom in a fresh kernel.
All utility functions live in `model_utils_shared.py`, `model_utils_clustering.py`, `model_utils_pairwise.py`, and `model_utils_seed.py`.